In [1]:
import torch                    # Tensor 데이터의 타입으로 파싱하기 위해 로드 
import torch.nn as nn           # torch안에 nn부분만 로드하여 nn 별칭으로 사용(nn -> 기본 뼈대)
import torch.optim as optim     # 옵티마이저(기울기의 변화를 주는 기능)

In [4]:
import pandas as pd

In [2]:
# 데이터셋을 하나 생성 
# 독립, 종속 데이터를 tensor로 생성 
# 독립 변수 -> sklearn을 이용한 ML에서는 2차원 데이터 
x = torch.tensor( [ [1.0], [2.0], [3.0], [4.0] ] )
# 종속 변수 -> sklearn을 이용한 ML에서는 1차원 -> torch에서는 2차원으로 생성 
# 종속 변수는 독립 변수에서 2를 곱하고 1를 더한 값
y = torch.tensor( [ [3.0], [5.0], [7.0], [9.0] ] )

In [5]:
pd.DataFrame([ [1.0], [2.0], [3.0], [4.0] ])

,0
0,1.0
1,2.0
2,3.0
3,4.0


In [3]:
type(x)

torch.Tensor

In [6]:
# sklearn에서 모델을 생성한다면 -> 가중치가 2이고 절편이 1인 규칙을 찾는 LinearRegression을 이용하여 예측 가능

# torch을 이용한 단순 선형 회귀 
# 순전파 ( 모델 학습 -> 예측 )
    # class 클래스명( 부모클래스 ):  --> 부모클래스의 기능을 상속 받아서 클래스를 선언
class LinearReg(nn.Module):

    # torch의 모듈을 이용한 클래스 생성시 2개의 함수를 필요로 선언 (생성자 함수, forward 함수)

    # 생성자 함수 
    def __init__(self):
        # self : 자기 자신( 클래스를 생성할때 저장이 되는 위치 )
        # super() : 부모 클래스(nn.Module)를 의미
        super(LinearReg, self).__init__()       # 부모 클래스의 생성자 함수를 실행

        # 선형 회귀 모델을 이용
        # nn 모듈 안에 Linear() 모델은 
        # 첫번째 인자값 : 입력 데이터(독립 변수)의 차원(피쳐)의 수 
        # 두번째 인자값 : 출력 데이터의 차원(피쳐)의 수
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)


In [7]:
# 클래스 생성 -> 회귀 모델을 생성
model = LinearReg()

In [8]:
# 손실 함수 
criterion = nn.MSELoss()

In [9]:
# 옵티마이저 설정 -> 가중치를 업데이트 
# 어떤 모델의 파라미터를 설정할것인가? -> 첫번째 인자
# lr 매개변수 -> 경사 하강법의 보폭 
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [10]:
# 순전파 (생성된 모델을 호출하면 -> forward() 함수를 호출하도록 nn.Module에서 설정이 되어있음)
pred = model(x)
# LinearReg 클래스 안에 forward함수를 호출하여 독립변수(x)를 인자값으로 사용한다.

# 손실함수 
loss = criterion(pred, y)

# 기울기를 초기화 
optimizer.zero_grad()

# 역전파 (자동 미분) -> 데이터가 있는 쪽으로 방향을 제시한다 -> 네비게이션
loss.backward()

# 가중치를 업데이트 ( 파라미터(모델) 수정 )
optimizer.step()

# loss 값을 확인 
print(loss)

tensor(27.3975, grad_fn=<MseLossBackward0>)


In [11]:
# DL 모델은 반복 학습이 기본 설정 -> 학습모드를 평가모드 전환
# eval() : 모델을 평가모드로 전환
# train() : 모델을 학습모드로 전환
model.eval()

# 예측, 평가 ( 메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화  )
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[0.4341],
        [1.3488],
        [2.2635],
        [3.1782]])
tensor(19.0606)


In [12]:
# 반복 학습을 통해서 가중치와 편향을 변화 시킨다. 
epochs = 200
model.train()
for epoch in range(epochs):
    # 순전파
    pred = model(x)
    # 손실 함수 
    loss = criterion(pred, y)
    # 기울기 초기화 
    optimizer.zero_grad()
    # 자동 미분(역전파) -> 가중치의 방향을 제시 
    loss.backward()
    # 가중치를 업데이트 
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        # 반복횟수가 20회마다 출력 
        print(f" Epoch : [ {epoch+1}, 200 ], Loss : {round(loss.item(), 6)} ")


 Epoch : [ 20, 200 ], Loss : 0.166119 
 Epoch : [ 40, 200 ], Loss : 0.131188 
 Epoch : [ 60, 200 ], Loss : 0.116351 
 Epoch : [ 80, 200 ], Loss : 0.1032 
 Epoch : [ 100, 200 ], Loss : 0.091536 
 Epoch : [ 120, 200 ], Loss : 0.081191 
 Epoch : [ 140, 200 ], Loss : 0.072014 
 Epoch : [ 160, 200 ], Loss : 0.063875 
 Epoch : [ 180, 200 ], Loss : 0.056656 
 Epoch : [ 200, 200 ], Loss : 0.050252 


In [13]:
model.eval()

# 예측, 평가 ( 메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화  )
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[2.6391],
        [4.8251],
        [7.0111],
        [9.1971]])
tensor(0.0500)


In [14]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

In [15]:
data = fetch_california_housing()

x = data['data']
y= data['target']

print(x.shape, y.shape)

(20640, 8) (20640,)


In [ ]:
import pandas as pd 
# csv 파일을 이용해서 x,y 생성 할때

# df = pd.read_csv("../data/california.csv")

# x = df.drop('target', axis=1).values
# y = df['target'].values

In [21]:
# 1차원 데이터를 2차원으로 변경 
y = y.reshape(-1, 1)

In [22]:
# 학습, 평가 데이터로 데이터 분할 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [24]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [26]:
y_train_tensor

tensor([[1.0300],
        [3.8210],
        [1.7260],
        ...,
        [2.2210],
        [2.8350],
        [3.2500]])